# Census of the 256×256 configuration

This notebook describes the 256×256 configuration of PASSAGE v1.0 in full. It reads the
metadata of every core scene in the four splits and answers four questions:

1. How many scenes does each split hold, and from how many source tiles?
2. Where are the scenes, and how much of each crop do the obstacles cover?
3. How much relief does the terrain show?
4. Do the stored reference routes hold when the exact solver runs again?

The first two answers and the route outcomes cover the whole configuration. Relief comes from
the 64 preview scenes of each split, because it needs the decoded terrain. The last section
solves a small sample of validation tasks again and compares the result with the stored labels.

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. The census reads about six minutes of metadata; the solver check takes under a minute.

In [ ]:
resolution = 256
census_limit = None  # None reads every scene; a positive number reads only the first rows of each split
relief_sample = 64  # preview scenes decoded per split to measure relief
check_tasks = 16  # validation scenes solved again, six routes each
max_expansions = resolution ** 2  # one expansion per cell lets every exact search finish
output_dir = "notebook-artifacts/256x256/dataset"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "2259dd94e2de28ab4a56c270247a7e347ec8f133"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))

## 1. Scenes per split

The census reads only the `id` and `metadata` columns, so it never decodes a tensor. For every
scene it records the split, the crop centre, the source tile, the obstacle coverage and the
status of the six stored routes. A metadata mismatch is a row whose recorded resolution or
split disagrees with the configuration it was read from.

In [ ]:
from collections import Counter

SPLITS = ("train", "calibration", "validation", "test")
SOLVERS = {"astar_elevation": "elevation", "astar_slope": "slope_uphill", "astar_energy": "energy"}

columns = {name: [] for name in ("split", "lat", "lon", "obstacle_fraction", "metadata_ok")}
tiles = {split: set() for split in SPLITS}
outcomes = Counter()
for split in SPLITS:
    for row in source.iter_rows(resolution, split, columns=["id", "metadata"], samples=False,
                                limit=census_limit):
        meta = row["metadata"]
        centre = meta["crop"]["center"]
        columns["split"].append(split)
        columns["lat"].append(centre["lat"])
        columns["lon"].append(centre["lon"])
        columns["obstacle_fraction"].append(meta["obstacles"]["ratio_actual"])
        columns["metadata_ok"].append(bool(row["id"]) and meta["resolution"] == resolution
                                      and meta["split"] == split)
        tile = meta["tile"]["source"]
        tiles[split].add((tile["lat"], tile["lon"]))
        for solver, objective in SOLVERS.items():
            for condition in ("free", "obstacles"):
                outcomes[(split, objective, condition, meta["paths"][solver][condition]["status"])] += 1
scenes = pd.DataFrame(columns)

per_split = pd.DataFrame({
    "Split": SPLITS,
    "Scenes": [int((scenes.split == s).sum()) for s in SPLITS],
    "Source tiles": [len(tiles[s]) for s in SPLITS],
    "Mean obstacle coverage (%)": [100 * scenes.obstacle_fraction[scenes.split == s].mean() for s in SPLITS],
    "Metadata mismatches": [int((~scenes.metadata_ok[scenes.split == s]).sum()) for s in SPLITS],
})
per_split.insert(2, "Share (%)", 100 * per_split["Scenes"] / per_split["Scenes"].sum())
show(per_split, f"Table 1. Core scenes of the {resolution}×{resolution} configuration per split.", "{:,.1f}")

The split shares follow the frozen 90 / 2 / 4 / 4 % policy. Each source tile belongs to one split
only, so the tile counts of the four splits never overlap. Obstacle coverage is the share of cells
that the obstacles block; the generator draws a target between 0 and 30 % for each scene.

## 2. Where the scenes are

Each point of Figure 1 is the centre of one crop. The test split is the South America hold-out;
the other splits share the remaining land by seeded 5°×5° blocks. Large splits are thinned to
15,000 points for drawing only; all statistics use every scene.

In [ ]:
fig, (ax_map, ax_cov) = plt.subplots(1, 2, figsize=(11, 4.3), gridspec_kw={"width_ratios": [1.7, 1]})
fig.subplots_adjust(top=0.82, wspace=0.18)
for split in SPLITS:
    part = scenes[scenes.split == split]
    if len(part) > 15_000:
        part = part.sample(15_000, random_state=0)
    ax_map.scatter(part.lon, part.lat, s=2.5, color=SPLIT_COLORS[split], alpha=0.55, linewidths=0,
                   label=split, rasterized=True)
ax_map.set(xlim=(-180, 180), ylim=(-60, 85), xlabel="Longitude (°)", ylabel="Latitude (°)",
           title="Crop centres by split")
ax_map.set_aspect("equal")
ax_map.grid(True, axis="both")
ax_map.legend(loc="lower left", markerscale=5, fontsize=8, ncol=4)

bins = np.linspace(0, 31, 32)
for split in SPLITS:
    ax_cov.hist(100 * scenes.obstacle_fraction[scenes.split == split], bins=bins, density=True,
                histtype="step", color=SPLIT_COLORS[split], linewidth=1.6, label=split)
ax_cov.set(xlabel="Obstacle coverage (% of cells)", ylabel="Density", title="Obstacle coverage")
ax_cov.legend(fontsize=8)
figure_title(fig, f"Where the {resolution}×{resolution} scenes are, and how much the obstacles cover",
             f"{len(scenes):,} core scenes · crop centres from metadata · PASSAGE v1.0")
figures = save_figure(fig, output_path, f"dataset-{resolution}-census")
plt.show()

*Figure 1. Left: the centre of every crop, coloured by split; the red test points cover South
America. Right: the distribution of obstacle coverage in each split. Similar curves mean that the
splits draw obstacles the same way.*

## 3. Terrain relief

Relief is the difference between the highest and the lowest cell of a crop, in metres. It needs
the decoded elevation channel, so it is measured on the preview scenes: the first 64 admitted
scenes of each split, also published in the `_samples` configuration.

In [ ]:
from passage.utils import decode_tensor_blob

relief_rows = []
for split in SPLITS:
    for row in source.iter_rows(resolution, split, columns=["id", "tensor", "metadata"], samples=True,
                                limit=relief_sample):
        calibration = row["metadata"]["calibration"]
        tensor = decode_tensor_blob(bytes(row["tensor"]), (resolution, resolution, 3))
        elevation = tensor[..., 0].astype(np.float64) * (calibration["max"] - calibration["min"]) + calibration["min"]
        relief_rows.append({"split": split, "id": row["id"], "relief_m": float(np.ptp(elevation)),
                            "mean_elevation_m": float(elevation.mean())})
relief = pd.DataFrame(relief_rows)

relief_table = relief.groupby("split").agg(
    Scenes=("relief_m", "size"),
    **{"Median relief (m)": ("relief_m", "median"),
       "10th percentile (m)": ("relief_m", lambda s: s.quantile(0.1)),
       "90th percentile (m)": ("relief_m", lambda s: s.quantile(0.9)),
       "Mean elevation (m)": ("mean_elevation_m", "mean")}).reindex(SPLITS).reset_index(names="Split")
show(relief_table, "Table 2. Terrain relief of the preview scenes.", "{:,.0f}")

fig, ax = plt.subplots(figsize=(6.4, 3.4))
fig.subplots_adjust(top=0.78)
parts = ax.boxplot([relief.relief_m[relief.split == s] for s in SPLITS], tick_labels=SPLITS,
                   widths=0.5, patch_artist=True, medianprops={"color": INK, "linewidth": 1.4},
                   flierprops={"markersize": 3, "markeredgecolor": MUTED})
for patch, split in zip(parts["boxes"], SPLITS):
    patch.set(facecolor=SPLIT_COLORS[split], alpha=0.35, edgecolor=SPLIT_COLORS[split])
ax.set_yscale("log")
ax.set(ylabel="Relief (m, log scale)")
figure_title(fig, "Terrain relief per split", f"{len(relief)} preview scenes · {resolution}×{resolution}")
figures += save_figure(fig, output_path, f"dataset-{resolution}-relief")
plt.show()

*Figure 2. Relief of the preview scenes in each split, on a log scale. Boxes span the middle half of
the scenes; the line is the median. Starts favour the elevation extremes of their tile, so many
crops sit on mountains or along coasts.*

## 4. Stored route outcomes

Each scene stores six routes: three objectives, each solved on free terrain and with the obstacles
blocked. Every released route has the status `ok`, an exact optimum. Table 3 counts the exact routes
of each objective and condition over the whole configuration and checks the totals against the
release receipt published with the data.

In [ ]:
counts = (pd.Series(outcomes).rename_axis(["split", "objective", "condition", "status"])
          .groupby(["objective", "condition", "status"]).sum().unstack("status", fill_value=0))
if "ok" not in counts:
    counts["ok"] = 0
outcome_table = counts.reset_index().rename(columns={"objective": "Objective", "condition": "Condition",
                                                      "ok": "Exact routes"})
outcome_table["Objective"] = outcome_table["Objective"].map(OBJECTIVE_LABELS)
outcome_table["Scenes"] = counts.sum(axis=1).to_numpy()
outcome_table["All exact"] = outcome_table["Exact routes"] == outcome_table["Scenes"]
show(outcome_table[["Objective", "Condition", "Scenes", "Exact routes", "All exact"]],
     "Table 3. Exact routes per objective and condition, all splits together.", "{:,.0f}")

exact_total = int(counts["ok"].sum())
variant_total = int(counts.to_numpy().sum())
facts_file = Path(source.root or "") / "release-receipts/publication-facts.json"
receipt_check = "not checked"
if census_limit is None and facts_file.is_file():
    facts = json.loads(facts_file.read_text())["resolutions"][str(resolution)]
    matches = (facts["successful_route_variants"] == exact_total == variant_total == 6 * len(scenes)
               and facts["proved_infeasible_variants"] == 0)
    receipt_check = "matches" if matches else "differs from"
display(Markdown(f"The census counts **{exact_total:,} exact routes** for **{variant_total:,} route "
                 f"variants**, six for each of the {len(scenes):,} scenes. This total {receipt_check} "
                 "the release receipt."))

Without obstacles, every cell connects to every other cell. With obstacles, the generator checks
the same eight-neighbour graph before it solves and rejects a crop whose obstacles disconnect the
start from the goal, so every stored route joins its endpoints.

## 5. Solving a sample of tasks again

A stored label is only as good as the solver that produced it. This section takes the first
`check_tasks` validation scenes, solves their six variants again with the exact float64 A*
reference of the code repository, and compares the result with the stored route. The search budget
equals the number of cells, so every search can finish. Ties are possible: two different routes can
share the optimal cost, so the check compares costs first and route geometry second.

In [ ]:
from passage.notebook_reports import classical_diagnostics_report

check = classical_diagnostics_report(source, resolution, budget=check_tasks, max_expansions=max_expansions)
rows = []
for record in check["records"]:
    exact = record["reference_status"] == "exact"
    rows.append({
        "Scene": record["sample_id"], "Objective": OBJECTIVE_LABELS.get(record["cost_model"], record["cost_model"]),
        "Condition": record["task_variant"], "Search outcome": record["reference_status"],
        "Expansions": int(record["reference_expansions"] or 0),
        "Stored cost": record["stored_cost"], "Re-solved cost": record["optimal_cost"],
        "Same cost": bool(exact and np.isclose(record["stored_cost"], record["optimal_cost"], rtol=1e-7, atol=1e-6)),
        "Same route": bool(exact and [tuple(p) for p in record["stored_path"]] == [tuple(p) for p in record["optimal_path"]]),
        "Legal": bool(record["legal"]),
    })
check_table = pd.DataFrame(rows)
show(check_table, f"Table 4. Stored routes against a fresh exact solve, {check_tasks} validation scenes.", "{:,.1f}")

A search outcome of `exact` means that the solver reached the goal and proved the optimum. `no_path`
would mean a proved absence of route, which would contradict the stored label. Any other outcome
would mean that the search stopped at its budget, which says nothing about the stored label.

## Summary

In [ ]:
n_exact = int((check_table["Search outcome"] == "exact").sum())
n_same_cost = int(check_table["Same cost"].sum())
n_same_route = int(check_table["Same route"].sum())
n_mismatch = int((~scenes.metadata_ok).sum())
display(Markdown("\n".join([
    f"- The {resolution}×{resolution} configuration holds **{len(scenes):,} core scenes** drawn from "
    f"**{len(set().union(*tiles.values())):,} source tiles**; {n_mismatch:,} rows carry metadata that "
    "disagree with their configuration.",
    f"- Obstacles cover {100 * scenes.obstacle_fraction.mean():.1f} % of the cells on average; the median "
    f"relief of the preview scenes is {relief.relief_m.median():,.0f} m.",
    f"- The stored labels hold {exact_total:,} exact routes for {variant_total:,} route variants "
    f"(six per scene); this {receipt_check} the release receipt.",
    f"- Solved again, {n_exact} of {len(check_table)} variants reached a proved optimum; {n_same_cost} match "
    f"the stored cost and {n_same_route} also follow the same cells.",
])))
report = {"resolution": resolution, "census_limit": census_limit, "per_split": per_split.to_dict("records"),
          "relief": relief_table.to_dict("records"), "route_outcomes": outcome_table.to_dict("records"),
          "receipt_check": receipt_check, "solver_check": check_table.to_dict("records"),
          "max_expansions": max_expansions, "figures": [relative(f) for f in figures]}

In [ ]:
report_file = write_report(report, output_path, f"dataset-{resolution}.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*